# Deep Dive: Correlation Analysis for NFL Trading Features

This notebook provides an in-depth correlation analysis of features extracted from NFL games and Kalshi price data. We'll explore:

1. **Feature-Target Correlations**: Which features best predict price movements
2. **Feature-Feature Correlations**: Understanding relationships between different feature types
3. **Time-Lagged Correlations**: How features correlate with future price movements
4. **Conditional Correlations**: How correlations change under different game conditions
5. **Rolling Correlations**: How feature relationships evolve over time
6. **Statistical Significance**: Which correlations are statistically meaningful

In [ ]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Import our feature engineering modules
from src.nfl_trading.features import (
    FeaturePipeline, 
    FeatureConfig, 
    FeatureValidator, 
    FeatureVisualizer
)

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (15, 10)

print("Libraries loaded successfully!")

## 1. Data Generation and Feature Engineering

Let's generate a more comprehensive dataset for detailed correlation analysis.

In [ ]:
def create_comprehensive_nfl_data(n_games=3):
    """Create multiple games worth of NFL data for robust analysis."""
    all_plays = []
    
    for game_num in range(n_games):
        base_time = datetime(2025, 9, 4 + game_num * 7, 20, 20)  # Weekly games
        
        current_time = base_time
        current_quarter = 1
        time_remaining = 15 * 60
        dal_score = 0
        phi_score = 0
        possession = 'DAL' if game_num % 2 == 0 else 'PHI'
        field_position = 25
        down = 1
        distance = 10
        
        # Game scenario variations
        if game_num == 0:  # Close game
            score_variance = 0.5
        elif game_num == 1:  # High scoring
            score_variance = 1.5
        else:  # Defensive game
            score_variance = 0.3
        
        play_types = ['pass', 'run', 'punt', 'field_goal', 'touchdown', 'interception', 'sack']
        
        for i in range(150):  # 150 plays per game
            # Adjust probabilities based on game situation
            if current_quarter >= 4 and abs(dal_score - phi_score) > 7:
                # Passing game when behind
                probs = [0.6, 0.2, 0.08, 0.03, 0.04, 0.03, 0.02]
            else:
                # Balanced offense
                probs = [0.4, 0.35, 0.1, 0.05, 0.04, 0.03, 0.03]
            
            play_type = np.random.choice(play_types, p=probs)
            
            # Simulate play outcome with more realistic logic
            if play_type == 'touchdown':
                yards_gained = max(1, 100 - field_position)
                if possession == 'DAL':
                    dal_score += 7
                else:
                    phi_score += 7
                possession = 'PHI' if possession == 'DAL' else 'DAL'
                field_position = 25
                down = 1
                distance = 10
            elif play_type == 'field_goal':
                yards_gained = 0
                if possession == 'DAL':
                    dal_score += 3
                else:
                    phi_score += 3
                possession = 'PHI' if possession == 'DAL' else 'DAL'
                field_position = 25
                down = 1
                distance = 10
            elif play_type == 'punt':
                yards_gained = 0
                possession = 'PHI' if possession == 'DAL' else 'DAL'
                field_position = np.random.randint(15, 45)
                down = 1
                distance = 10
            elif play_type == 'interception':
                yards_gained = -np.random.randint(0, 8)
                possession = 'PHI' if possession == 'DAL' else 'DAL'
                field_position = np.random.randint(20, 80)
                down = 1
                distance = 10
            elif play_type == 'sack':
                yards_gained = -np.random.randint(3, 12)
                field_position = max(1, field_position + yards_gained)
                down = min(4, down + 1)
                distance = max(1, distance - yards_gained)
            else:  # pass or run
                # More realistic yardage distribution
                if play_type == 'pass':
                    yards_gained = np.random.choice(
                        range(-5, 40), 
                        p=np.array([0.05, 0.05, 0.1, 0.1, 0.15, 0.15, 0.1, 0.08, 0.06, 0.04, 0.03, 0.02, 0.015, 0.01, 0.005] + [0.002]*30)
                    )
                else:  # run
                    yards_gained = np.random.choice(
                        range(-3, 25), 
                        p=np.array([0.1, 0.1, 0.15, 0.2, 0.15, 0.1, 0.08, 0.05, 0.03, 0.02, 0.01, 0.005] + [0.002]*16)
                    )
                
                field_position = min(100, max(1, field_position + yards_gained))
                
                if yards_gained >= distance:
                    down = 1
                    distance = min(10, 100 - field_position)
                else:
                    down += 1
                    distance = max(1, distance - yards_gained)
                    
                if down > 4:
                    possession = 'PHI' if possession == 'DAL' else 'DAL'
                    field_position = 100 - field_position
                    down = 1
                    distance = 10
            
            all_plays.append({
                'game_id': f'game_{game_num}',
                'timestamp': current_time,
                'play_type': play_type,
                'possession_team': possession,
                'field_position': field_position,
                'score_home': phi_score,
                'score_away': dal_score,
                'quarter': current_quarter,
                'time_remaining': time_remaining,
                'down': down,
                'distance': distance,
                'yards_gained': yards_gained,
                'home_team': 'PHI',
                'away_team': 'DAL'
            })
            
            # Update game time
            play_duration = np.random.randint(25, 45)
            current_time += timedelta(seconds=play_duration)
            time_remaining -= play_duration
            
            # Handle quarter changes
            if time_remaining <= 0 and current_quarter < 4:
                current_quarter += 1
                time_remaining = 15 * 60
    
    return pd.DataFrame(all_plays)

def create_realistic_price_data(nfl_data):
    """Create realistic price data with stronger correlations to game events."""
    prices = []
    
    # Group by game for consistent pricing
    for game_id, game_plays in nfl_data.groupby('game_id'):
        base_price = np.random.uniform(0.35, 0.65)  # Random starting probability
        
        start_time = game_plays['timestamp'].min() - timedelta(minutes=30)
        end_time = game_plays['timestamp'].max() + timedelta(minutes=30)
        
        current_time = start_time
        current_price = base_price
        
        while current_time <= end_time:
            # Find nearby NFL events
            nearby_plays = game_plays[
                (game_plays['timestamp'] >= current_time - timedelta(minutes=1)) &
                (game_plays['timestamp'] <= current_time + timedelta(minutes=1))
            ]
            
            # Base random walk
            price_change = np.random.randn() * 0.008
            
            # Strong event impacts
            for _, play in nearby_plays.iterrows():
                score_diff = play['score_away'] - play['score_home']  # DAL - PHI
                time_factor = 1.0 + (play['quarter'] - 1) * 0.2  # Later quarters matter more
                
                if play['possession_team'] == 'DAL':
                    if 'touchdown' in play['play_type']:
                        price_change += 0.08 * time_factor
                    elif 'field_goal' in play['play_type']:
                        price_change += 0.03 * time_factor
                    elif play['yards_gained'] > 20:  # Big play
                        price_change += 0.015 * time_factor
                    elif 'sack' in play['play_type'] or 'interception' in play['play_type']:
                        price_change -= 0.04 * time_factor
                    elif play['yards_gained'] < -5:  # Big loss
                        price_change -= 0.02 * time_factor
                else:  # PHI possession
                    if 'touchdown' in play['play_type']:
                        price_change -= 0.08 * time_factor
                    elif 'field_goal' in play['play_type']:
                        price_change -= 0.03 * time_factor
                    elif play['yards_gained'] > 20:
                        price_change -= 0.015 * time_factor
                    elif 'sack' in play['play_type'] or 'interception' in play['play_type']:
                        price_change += 0.04 * time_factor
                
                # Score differential impact
                if abs(score_diff) > 14:  # Blowout situation
                    price_change *= 0.5  # Reduced impact
                elif abs(score_diff) <= 3:  # Close game
                    price_change *= 1.5  # Increased impact
            
            current_price = np.clip(current_price + price_change, 0.01, 0.99)
            
            # Create realistic OHLCV
            volatility = 0.003 + abs(price_change) * 0.5
            high_price = current_price + abs(np.random.randn() * volatility)
            low_price = current_price - abs(np.random.randn() * volatility)
            open_price = current_price + np.random.randn() * volatility * 0.5
            
            # Volume increases with volatility
            base_volume = 8000
            volume_multiplier = 1 + abs(price_change) * 10
            volume = int(base_volume * volume_multiplier * (0.8 + np.random.random() * 0.4))
            
            prices.append({
                'game_id': game_id,
                'timestamp': current_time,
                'close_price': current_price,
                'open_price': np.clip(open_price, 0.01, 0.99),
                'high_price': np.clip(max(high_price, current_price, open_price), 0.01, 0.99),
                'low_price': np.clip(min(low_price, current_price, open_price), 0.01, 0.99),
                'volume': volume,
                'bid_price': np.clip(current_price - 0.005, 0.01, 0.99),
                'ask_price': np.clip(current_price + 0.005, 0.01, 0.99)
            })
            
            current_time += timedelta(seconds=30)  # 30-second intervals
    
    return pd.DataFrame(prices)

# Generate comprehensive dataset
print("Generating comprehensive multi-game dataset...")
nfl_data = create_comprehensive_nfl_data(n_games=3)
price_data = create_realistic_price_data(nfl_data)

print(f"Generated {len(nfl_data)} NFL plays across {nfl_data['game_id'].nunique()} games")
print(f"Generated {len(price_data)} price points")
print(f"Time range: {nfl_data['timestamp'].min()} to {nfl_data['timestamp'].max()}")

## 2. Feature Engineering with Enhanced Configuration

In [ ]:
# Enhanced configuration for correlation analysis
config = FeatureConfig(
    enable_game_features=True,
    enable_technical_features=True,
    enable_momentum_features=True,
    feature_windows=[3, 5, 10, 20],  # Multiple time windows
    scaling_method='standard',
    imputation_method='mean',
    handle_outliers=True,
    enable_feature_selection=False,
    min_data_points=20
)

# Initialize and run pipeline
pipeline = FeaturePipeline(config)

print("Running enhanced feature engineering...")
result = pipeline.fit_transform(
    nfl_data=nfl_data,
    price_data=price_data,
    team_focus='DAL'
)

print(f"\nFeatures extracted: {len(result.feature_names)}")
print(f"Samples: {len(result.features_df)}")
print(f"Targets: {result.target_columns}")

# Clean data for correlation analysis
df = result.features_df.copy()
feature_cols = [col for col in result.feature_names if col in df.columns]
target_cols = [col for col in result.target_columns if col in df.columns]

print(f"\nReady for correlation analysis with {len(feature_cols)} features and {len(target_cols)} targets")

## 3. Comprehensive Correlation Analysis

In [ ]:
def calculate_comprehensive_correlations(df, feature_cols, target_cols):
    """Calculate comprehensive correlation statistics."""
    results = {}
    
    for target in target_cols:
        if target not in df.columns:
            continue
            
        target_results = []
        
        for feature in feature_cols:
            if feature not in df.columns:
                continue
                
            # Remove NaN values
            valid_mask = ~(df[feature].isna() | df[target].isna())
            
            if valid_mask.sum() < 10:
                continue
                
            x = df[feature][valid_mask]
            y = df[target][valid_mask]
            
            if x.std() == 0 or y.std() == 0:
                continue
            
            # Pearson correlation
            pearson_corr, pearson_p = pearsonr(x, y)
            
            # Spearman correlation (rank-based, captures non-linear relationships)
            spearman_corr, spearman_p = spearmanr(x, y)
            
            target_results.append({
                'feature': feature,
                'pearson_corr': pearson_corr,
                'pearson_p_value': pearson_p,
                'spearman_corr': spearman_corr,
                'spearman_p_value': spearman_p,
                'abs_pearson': abs(pearson_corr),
                'abs_spearman': abs(spearman_corr),
                'n_samples': len(x),
                'pearson_significant': pearson_p < 0.05,
                'spearman_significant': spearman_p < 0.05
            })
        
        results[target] = pd.DataFrame(target_results)
    
    return results

# Calculate correlations
print("Calculating comprehensive correlations...")
correlation_results = calculate_comprehensive_correlations(df, feature_cols, target_cols)

# Display results for main target
if correlation_results:
    main_target = list(correlation_results.keys())[0]
    main_df = correlation_results[main_target]
    
    print(f"\n=== CORRELATION ANALYSIS FOR {main_target} ===")
    print(f"Total features analyzed: {len(main_df)}")
    print(f"Statistically significant (p < 0.05): {main_df['pearson_significant'].sum()}")
    
    # Top correlations by Pearson
    top_pearson = main_df.nlargest(15, 'abs_pearson')
    print("\nTop 15 Features by Absolute Pearson Correlation:")
    display_cols = ['feature', 'pearson_corr', 'pearson_p_value', 'pearson_significant']
    print(top_pearson[display_cols].to_string(index=False, float_format='%.4f'))
    
    # Top correlations by Spearman (non-linear relationships)
    top_spearman = main_df.nlargest(15, 'abs_spearman')
    print("\nTop 15 Features by Absolute Spearman Correlation:")
    display_cols = ['feature', 'spearman_corr', 'spearman_p_value', 'spearman_significant']
    print(top_spearman[display_cols].to_string(index=False, float_format='%.4f'))

## 4. Feature Category Correlation Analysis

In [ ]:
def analyze_feature_categories(correlation_df):
    """Analyze correlations by feature category."""
    categories = {
        'Game State': ['game', 'drive', 'situation', 'down', 'field', 'quarter', 'time_'],
        'Technical Indicators': ['sma', 'ema', 'rsi', 'macd', 'bb_', 'volatility', 'atr'],
        'Momentum': ['momentum'],
        'Volume': ['volume', 'obv', 'vpt'],
        'Price Patterns': ['doji', 'hammer', 'shooting', 'gap', 'local'],
        'Time Windows': ['rolling', 'change_', 'pct_change'],
        'Team Specific': ['team_']
    }
    
    category_stats = []
    
    for category, keywords in categories.items():
        # Find features in this category
        category_features = correlation_df[
            correlation_df['feature'].str.contains('|'.join(keywords), case=False, na=False)
        ]
        
        if not category_features.empty:
            stats = {
                'category': category,
                'n_features': len(category_features),
                'avg_abs_pearson': category_features['abs_pearson'].mean(),
                'max_abs_pearson': category_features['abs_pearson'].max(),
                'avg_abs_spearman': category_features['abs_spearman'].mean(),
                'max_abs_spearman': category_features['abs_spearman'].max(),
                'significant_features': category_features['pearson_significant'].sum(),
                'significance_rate': category_features['pearson_significant'].mean()
            }
            category_stats.append(stats)
    
    return pd.DataFrame(category_stats).sort_values('avg_abs_pearson', ascending=False)

if correlation_results:
    main_df = correlation_results[main_target]
    
    # Analyze by category
    category_analysis = analyze_feature_categories(main_df)
    
    print("\n=== FEATURE CATEGORY ANALYSIS ===")
    print(category_analysis.to_string(index=False, float_format='%.4f'))
    
    # Visualization of category performance
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    # Average correlation by category
    category_analysis.plot(x='category', y='avg_abs_pearson', kind='bar', ax=ax1, color='skyblue')
    ax1.set_title('Average Absolute Pearson Correlation by Category', fontweight='bold')
    ax1.set_ylabel('Average |Correlation|')
    ax1.tick_params(axis='x', rotation=45)
    ax1.grid(axis='y', alpha=0.3)
    
    # Max correlation by category
    category_analysis.plot(x='category', y='max_abs_pearson', kind='bar', ax=ax2, color='lightcoral')
    ax2.set_title('Maximum Absolute Pearson Correlation by Category', fontweight='bold')
    ax2.set_ylabel('Maximum |Correlation|')
    ax2.tick_params(axis='x', rotation=45)
    ax2.grid(axis='y', alpha=0.3)
    
    # Number of features by category
    category_analysis.plot(x='category', y='n_features', kind='bar', ax=ax3, color='lightgreen')
    ax3.set_title('Number of Features by Category', fontweight='bold')
    ax3.set_ylabel('Number of Features')
    ax3.tick_params(axis='x', rotation=45)
    ax3.grid(axis='y', alpha=0.3)
    
    # Significance rate by category
    category_analysis.plot(x='category', y='significance_rate', kind='bar', ax=ax4, color='gold')
    ax4.set_title('Statistical Significance Rate by Category', fontweight='bold')
    ax4.set_ylabel('Proportion Significant (p < 0.05)')
    ax4.tick_params(axis='x', rotation=45)
    ax4.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 5. Time-Lagged Correlation Analysis

In [ ]:
def calculate_lagged_correlations(df, features, target, max_lag=10):
    """Calculate correlations between features and future values of target."""
    lagged_results = []
    
    # Sort by timestamp for proper lagging
    if 'timestamp' in df.columns:
        df_sorted = df.sort_values('timestamp').reset_index(drop=True)
    else:
        df_sorted = df.copy()
    
    for lag in range(max_lag + 1):
        # Create lagged target
        if lag == 0:
            lagged_target = df_sorted[target]
        else:
            lagged_target = df_sorted[target].shift(-lag)
        
        lag_correlations = []
        
        for feature in features[:20]:  # Top 20 features to avoid clutter
            if feature not in df_sorted.columns:
                continue
                
            valid_mask = ~(df_sorted[feature].isna() | lagged_target.isna())
            
            if valid_mask.sum() < 10:
                continue
                
            x = df_sorted[feature][valid_mask]
            y = lagged_target[valid_mask]
            
            if x.std() == 0 or y.std() == 0:
                continue
            
            corr, p_value = pearsonr(x, y)
            
            lag_correlations.append({
                'feature': feature,
                'lag': lag,
                'correlation': corr,
                'abs_correlation': abs(corr),
                'p_value': p_value,
                'significant': p_value < 0.05
            })
        
        lagged_results.extend(lag_correlations)
    
    return pd.DataFrame(lagged_results)

if correlation_results:
    main_df = correlation_results[main_target]
    top_features = main_df.nlargest(20, 'abs_pearson')['feature'].tolist()
    
    print("\n=== TIME-LAGGED CORRELATION ANALYSIS ===")
    print("Calculating correlations with future price movements...")
    
    lagged_correlations = calculate_lagged_correlations(df, top_features, main_target, max_lag=10)
    
    if not lagged_correlations.empty:
        # Find features that predict future movements
        future_predictors = lagged_correlations[
            (lagged_correlations['lag'] > 0) & 
            (lagged_correlations['significant'] == True)
        ].nlargest(10, 'abs_correlation')
        
        print("\nTop 10 Features Predicting Future Price Movements:")
        display_cols = ['feature', 'lag', 'correlation', 'p_value']
        print(future_predictors[display_cols].to_string(index=False, float_format='%.4f'))
        
        # Visualize lagged correlations for top features
        plt.figure(figsize=(15, 10))
        
        # Select top 5 features for visualization
        top_5_features = main_df.nlargest(5, 'abs_pearson')['feature'].tolist()
        
        for i, feature in enumerate(top_5_features):
            feature_lags = lagged_correlations[lagged_correlations['feature'] == feature]
            if not feature_lags.empty:
                plt.plot(feature_lags['lag'], feature_lags['correlation'], 
                        marker='o', label=feature[:30], linewidth=2, markersize=6)
        
        plt.axhline(y=0, color='black', linestyle='-', alpha=0.3)
        plt.xlabel('Lag (periods into future)', fontsize=12)
        plt.ylabel('Correlation with Future Price Change', fontsize=12)
        plt.title('Time-Lagged Correlations: Current Features vs Future Price Changes', 
                 fontsize=14, fontweight='bold')
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        # Correlation decay analysis
        avg_correlation_by_lag = lagged_correlations.groupby('lag')['abs_correlation'].mean()
        
        plt.figure(figsize=(10, 6))
        plt.plot(avg_correlation_by_lag.index, avg_correlation_by_lag.values, 
                marker='o', linewidth=3, markersize=8, color='red')
        plt.xlabel('Lag (periods)', fontsize=12)
        plt.ylabel('Average Absolute Correlation', fontsize=12)
        plt.title('Correlation Decay: How Predictive Power Diminishes Over Time', 
                 fontsize=14, fontweight='bold')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        print(f"\nCorrelation decay analysis:")
        print(f"Immediate correlation (lag 0): {avg_correlation_by_lag.iloc[0]:.4f}")
        print(f"1-period ahead: {avg_correlation_by_lag.iloc[1]:.4f}")
        print(f"5-periods ahead: {avg_correlation_by_lag.iloc[5]:.4f}")
        print(f"10-periods ahead: {avg_correlation_by_lag.iloc[-1]:.4f}")

## 6. Conditional Correlation Analysis

In [ ]:
def analyze_conditional_correlations(df, feature_cols, target, conditions):
    """Analyze how correlations change under different conditions."""
    conditional_results = {}
    
    for condition_name, condition_func in conditions.items():
        try:
            # Apply condition
            condition_mask = condition_func(df)
            conditional_df = df[condition_mask]
            
            if len(conditional_df) < 20:  # Need minimum samples
                continue
            
            correlations = []
            
            for feature in feature_cols[:15]:  # Top 15 features
                if feature not in conditional_df.columns or target not in conditional_df.columns:
                    continue
                    
                valid_mask = ~(conditional_df[feature].isna() | conditional_df[target].isna())
                
                if valid_mask.sum() < 10:
                    continue
                    
                x = conditional_df[feature][valid_mask]
                y = conditional_df[target][valid_mask]
                
                if x.std() == 0 or y.std() == 0:
                    continue
                
                corr, p_value = pearsonr(x, y)
                
                correlations.append({
                    'feature': feature,
                    'correlation': corr,
                    'abs_correlation': abs(corr),
                    'p_value': p_value,
                    'n_samples': len(x)
                })
            
            if correlations:
                conditional_results[condition_name] = pd.DataFrame(correlations)
        
        except Exception as e:
            print(f"Error processing condition {condition_name}: {e}")
            continue
    
    return conditional_results

# Define game conditions for analysis
def create_game_conditions(df):
    """Create condition functions for different game situations."""
    conditions = {}
    
    # Check if we have the required columns
    if 'quarter' in df.columns:
        conditions['Early Game (Q1-Q2)'] = lambda x: x['quarter'] <= 2
        conditions['Late Game (Q3-Q4)'] = lambda x: x['quarter'] >= 3
        conditions['Final Quarter'] = lambda x: x['quarter'] == 4
    
    if 'score_differential' in df.columns:
        conditions['Close Game (≤7 pts)'] = lambda x: abs(x['score_differential']) <= 7
        conditions['Blowout (>14 pts)'] = lambda x: abs(x['score_differential']) > 14
    elif 'score_home' in df.columns and 'score_away' in df.columns:
        conditions['Close Game (≤7 pts)'] = lambda x: abs(x['score_home'] - x['score_away']) <= 7
        conditions['Blowout (>14 pts)'] = lambda x: abs(x['score_home'] - x['score_away']) > 14
    
    if 'volume' in df.columns:
        volume_median = df['volume'].median()
        conditions['High Volume'] = lambda x: x['volume'] > volume_median
        conditions['Low Volume'] = lambda x: x['volume'] <= volume_median
    
    if 'close_price' in df.columns:
        price_median = df['close_price'].median()
        conditions['High Price (>median)'] = lambda x: x['close_price'] > price_median
        conditions['Low Price (≤median)'] = lambda x: x['close_price'] <= price_median
    
    return conditions

if correlation_results:
    main_df = correlation_results[main_target]
    top_features = main_df.nlargest(15, 'abs_pearson')['feature'].tolist()
    
    print("\n=== CONDITIONAL CORRELATION ANALYSIS ===")
    
    conditions = create_game_conditions(df)
    
    if conditions:
        print(f"Analyzing correlations under {len(conditions)} different conditions...")
        
        conditional_results = analyze_conditional_correlations(df, top_features, main_target, conditions)
        
        if conditional_results:
            # Compare correlations across conditions
            comparison_data = []
            
            # Get baseline correlations (all data)
            baseline_correlations = {}
            for _, row in main_df.iterrows():
                if row['feature'] in top_features:
                    baseline_correlations[row['feature']] = row['abs_pearson']
            
            for condition_name, condition_df in conditional_results.items():
                avg_correlation = condition_df['abs_correlation'].mean()
                max_correlation = condition_df['abs_correlation'].max()
                n_samples = condition_df['n_samples'].iloc[0] if not condition_df.empty else 0
                
                comparison_data.append({
                    'condition': condition_name,
                    'avg_abs_correlation': avg_correlation,
                    'max_abs_correlation': max_correlation,
                    'n_samples': n_samples,
                    'n_features': len(condition_df)
                })
            
            comparison_df = pd.DataFrame(comparison_data)
            
            print("\nCorrelation Strength by Game Condition:")
            print(comparison_df.to_string(index=False, float_format='%.4f'))
            
            # Visualize conditional correlations
            plt.figure(figsize=(14, 8))
            
            x_pos = range(len(comparison_df))
            bars = plt.bar(x_pos, comparison_df['avg_abs_correlation'], 
                          color='steelblue', alpha=0.7, edgecolor='black')
            
            plt.xlabel('Game Condition', fontsize=12)
            plt.ylabel('Average Absolute Correlation', fontsize=12)
            plt.title('Feature-Price Correlations Under Different Game Conditions', 
                     fontsize=14, fontweight='bold')
            plt.xticks(x_pos, comparison_df['condition'], rotation=45, ha='right')
            
            # Add value labels on bars
            for bar, value in zip(bars, comparison_df['avg_abs_correlation']):
                plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                        f'{value:.3f}', ha='center', va='bottom', fontweight='bold')
            
            # Add sample size annotations
            for i, (bar, n_samples) in enumerate(zip(bars, comparison_df['n_samples'])):
                plt.text(bar.get_x() + bar.get_width()/2, 0.001,
                        f'n={n_samples}', ha='center', va='bottom', fontsize=9, alpha=0.7)
            
            plt.grid(axis='y', alpha=0.3)
            plt.tight_layout()
            plt.show()
            
            # Find conditions with strongest correlations
            best_condition = comparison_df.loc[comparison_df['avg_abs_correlation'].idxmax()]
            print(f"\nStrongest correlations found under: {best_condition['condition']}")
            print(f"Average correlation: {best_condition['avg_abs_correlation']:.4f}")
            print(f"Sample size: {best_condition['n_samples']}")
    else:
        print("No suitable conditions found for analysis.")

## 7. Rolling Correlation Analysis

In [ ]:
def calculate_rolling_correlations(df, features, target, window=50):
    """Calculate rolling correlations to see how relationships change over time."""
    if 'timestamp' not in df.columns:
        print("No timestamp column found for rolling analysis.")
        return None
    
    # Sort by timestamp
    df_sorted = df.sort_values('timestamp').reset_index(drop=True)
    
    rolling_results = []
    
    for i in range(window, len(df_sorted)):
        window_df = df_sorted.iloc[i-window:i]
        timestamp = df_sorted.iloc[i]['timestamp']
        
        for feature in features[:10]:  # Top 10 features to avoid clutter
            if feature not in window_df.columns or target not in window_df.columns:
                continue
                
            valid_mask = ~(window_df[feature].isna() | window_df[target].isna())
            
            if valid_mask.sum() < 10:
                continue
                
            x = window_df[feature][valid_mask]
            y = window_df[target][valid_mask]
            
            if x.std() == 0 or y.std() == 0:
                continue
            
            corr, _ = pearsonr(x, y)
            
            rolling_results.append({
                'timestamp': timestamp,
                'feature': feature,
                'rolling_correlation': corr,
                'abs_rolling_correlation': abs(corr)
            })
    
    return pd.DataFrame(rolling_results)

if correlation_results and 'timestamp' in df.columns:
    main_df = correlation_results[main_target]
    top_features = main_df.nlargest(10, 'abs_pearson')['feature'].tolist()
    
    print("\n=== ROLLING CORRELATION ANALYSIS ===")
    print("Analyzing how feature-price relationships evolve over time...")
    
    rolling_correlations = calculate_rolling_correlations(df, top_features, main_target, window=30)
    
    if rolling_correlations is not None and not rolling_correlations.empty:
        # Visualize rolling correlations for top features
        plt.figure(figsize=(16, 10))
        
        top_5_features = top_features[:5]
        
        for feature in top_5_features:
            feature_data = rolling_correlations[rolling_correlations['feature'] == feature]
            if not feature_data.empty:
                plt.plot(feature_data['timestamp'], feature_data['rolling_correlation'], 
                        label=feature[:25], linewidth=2, alpha=0.8)
        
        plt.axhline(y=0, color='black', linestyle='-', alpha=0.3)
        plt.xlabel('Time', fontsize=12)
        plt.ylabel('Rolling Correlation (30-period window)', fontsize=12)
        plt.title('Evolution of Feature-Price Correlations Over Time', 
                 fontsize=14, fontweight='bold')
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.grid(True, alpha=0.3)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
        
        # Correlation stability analysis
        stability_stats = []
        
        for feature in top_features:
            feature_data = rolling_correlations[rolling_correlations['feature'] == feature]
            if not feature_data.empty:
                correlations = feature_data['rolling_correlation']
                stability_stats.append({
                    'feature': feature,
                    'mean_correlation': correlations.mean(),
                    'std_correlation': correlations.std(),
                    'min_correlation': correlations.min(),
                    'max_correlation': correlations.max(),
                    'correlation_range': correlations.max() - correlations.min(),
                    'stability_score': 1 / (1 + correlations.std())  # Higher = more stable
                })
        
        stability_df = pd.DataFrame(stability_stats).sort_values('stability_score', ascending=False)
        
        print("\nFeature Correlation Stability Analysis:")
        print("(Higher stability score = more consistent correlation over time)")
        display_cols = ['feature', 'mean_correlation', 'std_correlation', 'correlation_range', 'stability_score']
        print(stability_df[display_cols].head(10).to_string(index=False, float_format='%.4f'))
        
        # Plot stability scores
        plt.figure(figsize=(12, 8))
        top_stable = stability_df.head(10)
        bars = plt.barh(range(len(top_stable)), top_stable['stability_score'], 
                       color='green', alpha=0.7)
        plt.yticks(range(len(top_stable)), [f[:30] for f in top_stable['feature']])
        plt.xlabel('Stability Score', fontsize=12)
        plt.title('Feature Correlation Stability Rankings', fontsize=14, fontweight='bold')
        plt.grid(axis='x', alpha=0.3)
        
        # Add stability score labels
        for i, bar in enumerate(bars):
            width = bar.get_width()
            plt.text(width + 0.01, bar.get_y() + bar.get_height()/2,
                    f'{width:.3f}', ha='left', va='center', fontweight='bold')
        
        plt.tight_layout()
        plt.show()
else:
    print("\nSkipping rolling correlation analysis (insufficient timestamp data).")

## 8. Summary and Key Insights

In [ ]:
print("=" * 80)
print("COMPREHENSIVE CORRELATION ANALYSIS SUMMARY")
print("=" * 80)

if correlation_results:
    main_df = correlation_results[main_target]
    
    print(f"\n📊 OVERALL STATISTICS:")
    print(f"• Target Variable: {main_target}")
    print(f"• Features Analyzed: {len(main_df)}")
    print(f"• Statistically Significant: {main_df['pearson_significant'].sum()} ({main_df['pearson_significant'].mean()*100:.1f}%)")
    print(f"• Strongest Correlation: {main_df['abs_pearson'].max():.4f}")
    print(f"• Average Correlation: {main_df['abs_pearson'].mean():.4f}")
    
    print(f"\n🏆 TOP PREDICTIVE FEATURES:")
    top_5 = main_df.nlargest(5, 'abs_pearson')
    for i, (_, row) in enumerate(top_5.iterrows(), 1):
        significance = "***" if row['pearson_p_value'] < 0.001 else "**" if row['pearson_p_value'] < 0.01 else "*" if row['pearson_p_value'] < 0.05 else ""
        print(f"{i}. {row['feature'][:50]}: {row['pearson_corr']:+.4f} {significance}")
    
    print(f"\n📈 FEATURE CATEGORY INSIGHTS:")
    if 'category_analysis' in locals():
        best_category = category_analysis.iloc[0]
        print(f"• Best Performing Category: {best_category['category']}")
        print(f"  - Average Correlation: {best_category['avg_abs_pearson']:.4f}")
        print(f"  - Features: {best_category['n_features']}")
        print(f"  - Significance Rate: {best_category['significance_rate']*100:.1f}%")
        
        print(f"\n• Category Rankings by Average Correlation:")
        for i, (_, row) in enumerate(category_analysis.head(5).iterrows(), 1):
            print(f"  {i}. {row['category']}: {row['avg_abs_pearson']:.4f}")
    
    print(f"\n⏰ TEMPORAL INSIGHTS:")
    if 'lagged_correlations' in locals() and not lagged_correlations.empty:
        future_predictors = lagged_correlations[
            (lagged_correlations['lag'] > 0) & 
            (lagged_correlations['significant'] == True)
        ]
        
        if not future_predictors.empty:
            best_predictor = future_predictors.nlargest(1, 'abs_correlation').iloc[0]
            print(f"• Best Future Predictor: {best_predictor['feature'][:40]}")
            print(f"  - Predicts {best_predictor['lag']} periods ahead")
            print(f"  - Correlation: {best_predictor['correlation']:+.4f}")
        
        if 'avg_correlation_by_lag' in locals():
            print(f"• Correlation Decay:")
            print(f"  - Immediate: {avg_correlation_by_lag.iloc[0]:.4f}")
            print(f"  - 1-period ahead: {avg_correlation_by_lag.iloc[1]:.4f}")
            print(f"  - 5-periods ahead: {avg_correlation_by_lag.iloc[5]:.4f}")
    
    print(f"\n🎯 CONDITIONAL INSIGHTS:")
    if 'conditional_results' in locals() and conditional_results:
        if 'comparison_df' in locals():
            best_condition = comparison_df.loc[comparison_df['avg_abs_correlation'].idxmax()]
            print(f"• Strongest Correlations Under: {best_condition['condition']}")
            print(f"  - Average Correlation: {best_condition['avg_abs_correlation']:.4f}")
            print(f"  - Sample Size: {best_condition['n_samples']}")
            
            worst_condition = comparison_df.loc[comparison_df['avg_abs_correlation'].idxmin()]
            print(f"• Weakest Correlations Under: {worst_condition['condition']}")
            print(f"  - Average Correlation: {worst_condition['avg_abs_correlation']:.4f}")
    
    print(f"\n⚖️ STABILITY INSIGHTS:")
    if 'stability_df' in locals() and not stability_df.empty:
        most_stable = stability_df.iloc[0]
        least_stable = stability_df.iloc[-1]
        print(f"• Most Stable Feature: {most_stable['feature'][:40]}")
        print(f"  - Stability Score: {most_stable['stability_score']:.4f}")
        print(f"  - Correlation Range: {most_stable['correlation_range']:.4f}")
        print(f"• Least Stable Feature: {least_stable['feature'][:40]}")
        print(f"  - Stability Score: {least_stable['stability_score']:.4f}")
        print(f"  - Correlation Range: {least_stable['correlation_range']:.4f}")
    
    print(f"\n🚀 KEY RECOMMENDATIONS:")
    recommendations = [
        f"Focus on features with correlation > {main_df['abs_pearson'].quantile(0.8):.3f} for initial models",
        f"Monitor {main_df['pearson_significant'].sum()} statistically significant features",
        "Use ensemble methods to combine different feature types",
        "Implement time-lagged features for prediction",
        "Consider conditional models for different game situations",
        "Prioritize stable features for robust trading strategies"
    ]
    
    for i, rec in enumerate(recommendations, 1):
        print(f"{i}. {rec}")
    
    print(f"\n💡 ACTIONABLE INSIGHTS:")
    insights = []
    
    # Feature type insights
    if 'category_analysis' in locals():
        best_cat = category_analysis.iloc[0]['category']
        insights.append(f"{best_cat} features show strongest predictive power")
    
    # Correlation strength insights
    strong_features = (main_df['abs_pearson'] > 0.1).sum()
    if strong_features > 5:
        insights.append(f"{strong_features} features show strong correlation (>0.1)")
    
    # Statistical significance insights
    sig_rate = main_df['pearson_significant'].mean()
    if sig_rate > 0.3:
        insights.append(f"High statistical significance rate ({sig_rate*100:.1f}%) indicates robust relationships")
    
    # Temporal insights
    if 'future_predictors' in locals() and not future_predictors.empty:
        insights.append(f"Features can predict price movements up to {future_predictors['lag'].max()} periods ahead")
    
    for insight in insights:
        print(f"• {insight}")

else:
    print("No correlation results available for analysis.")

print("\n" + "=" * 80)
print("DEEP DIVE CORRELATION ANALYSIS COMPLETED")
print("=" * 80)

## Conclusion

This comprehensive correlation analysis provides deep insights into the relationships between NFL game features and Kalshi price movements. The analysis reveals:

1. **Feature Effectiveness**: Different types of features (game state, technical, momentum) show varying predictive power
2. **Temporal Dynamics**: Feature-price relationships evolve over time and can predict future movements
3. **Conditional Relationships**: Correlations change significantly under different game conditions
4. **Stability Patterns**: Some features provide consistent predictive power while others are more volatile

These insights form the foundation for building robust NFL trading models that can adapt to different market and game conditions.